In [1]:
import numpy as np
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

Setting configurations for the model such as hyperparameters and the prepared splits of data from the preprocessing

In [7]:
# =========================================================
# --- Configuration ---
# =========================================================
VARIANT = "bivariate"  # or 'univariate', 'bivariate'
DATA_DIR = f"../preprocessing_scripts/preprocessed_classification_proper_split/w12/{VARIANT}"
OUTPUT_DIR = f"results/{VARIANT}_classification_proper_split"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIME_STEPS = 12
HORIZON = 4
EPOCHS = 100
BATCH_SIZE = 32
LR = 0.001
REG_LIMIT = 160  # µg/kg threshold (for reference only)


# =========================================================
# --- Load Datasets ---
# =========================================================
print(f"\n Loading preprocessed data from {DATA_DIR}...")

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
X_val   = np.load(os.path.join(DATA_DIR, "X_val.npy"))
y_val   = np.load(os.path.join(DATA_DIR, "y_val.npy"))
X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}")
num_features = X_train.shape[2]


 Loading preprocessed data from ../preprocessing_scripts/preprocessed_classification_proper_split/w12/bivariate...
Shapes -> X_train: (1984, 12, 2), y_train: (1984, 4)


In [8]:
# =========================================================
# --- Check for Class Imbalance ---
# =========================================================
print("\n Checking class distribution across horizons...")

for h in range(y_train.shape[1]):  # Loop over horizons
    unique, counts = np.unique(y_train[:, h], return_counts=True)
    total = counts.sum()
    proportions = {int(k): f"{v} ({v/total:.2%})" for k, v in zip(unique, counts)}
    print(f"Horizon t+{h+1}: {proportions}")

# Optionally, flatten across all horizons
y_train_flat = y_train.reshape(-1)
unique, counts = np.unique(y_train_flat, return_counts=True)
print("\nOverall class distribution (all horizons combined):")
for k, v in zip(unique, counts):
    print(f"  Class {k}: {v} ({v/len(y_train_flat):.2%})")



 Checking class distribution across horizons...
Horizon t+1: {0: '1577 (79.49%)', 1: '407 (20.51%)'}
Horizon t+2: {0: '1576 (79.44%)', 1: '408 (20.56%)'}
Horizon t+3: {0: '1575 (79.39%)', 1: '409 (20.61%)'}
Horizon t+4: {0: '1576 (79.44%)', 1: '408 (20.56%)'}

Overall class distribution (all horizons combined):
  Class 0: 6304 (79.44%)
  Class 1: 1632 (20.56%)


Model training

In [9]:
def build_lstm_classifier(input_shape, horizon):
    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(horizon, activation='sigmoid')  # 4 binary outputs (0–1)
    ])
    model.compile(
        optimizer=RMSprop(learning_rate=LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_lstm_classifier((TIME_STEPS, num_features), HORIZON)
model.summary()



# Flatten all horizons for weighting
y_train_flat = y_train.reshape(-1)
classes = np.unique(y_train_flat)

weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_flat)
class_weight_dict = {k: v for k, v in zip(classes, weights)}

print("Class weights:", class_weight_dict)


history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

c:\Users\jverduzco\anaconda3\envs\llm-harmful-algal-bloom\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,364 (75.64 KB)

 Trainable params: 19,364 (75.64 KB)

 Non-trainable params: 0 (0.00 B)

Class weights: {np.int64(0): np.float64(0.6294416243654822), np.int64(1): np.float64(2.4313725490196076)}
Epoch 1/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.1346 - loss: 0.5347 - val_accuracy: 0.0525 - val_loss: 0.3802
Epoch 2/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2223 - loss: 0.4481 - val_accuracy: 0.0756 - val_loss: 0.3844
Epoch 3/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1588 - loss: 0.4395 - val_accuracy: 0.0621 - val_loss: 0.3612
Epoch 4/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1714 - loss: 0.4254 - val_accuracy: 0.1354 - val_loss: 0.3611
Epoch 5/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2016 - loss: 0.4126 - val_accuracy: 0.0820 - val_loss: 0.3380
Epoch 6/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1996 - loss: 0.3997 - val_accuracy: 0.1545 - val_loss: 0.3226
Epoch 7/100
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2067 - loss: 0.3898 - val_accuracy: 0.1672 - val_loss: 0.3219
Epoc

Model evaluation on combined dataset

In [10]:
print("\n📈 Evaluating on test set...")
y_pred = model.predict(X_test)
y_pred_bin = (y_pred > 0.5).astype(int)

# --- Per-horizon metrics ---
for h in range(HORIZON):
    print(f"\n=== Horizon t+{h+1} ===")
    cm = confusion_matrix(y_test[:, h], y_pred_bin[:, h])
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_test[:, h], y_pred_bin[:, h], digits=3))

# =========================================================
# --- Save Model and Results ---
# =========================================================
model.save(os.path.join(OUTPUT_DIR, "lstm_classifier.keras"))
np.save(os.path.join(OUTPUT_DIR, "y_pred.npy"), y_pred)
np.save(os.path.join(OUTPUT_DIR, "y_pred_bin.npy"), y_pred_bin)


📈 Evaluating on test set...
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

=== Horizon t+1 ===
Confusion Matrix:
[[639   9]
 [ 30  64]]

Classification Report:
              precision    recall  f1-score   support

           0      0.955     0.986     0.970       648
           1      0.877     0.681     0.766        94

    accuracy                          0.947       742
   macro avg      0.916     0.833     0.868       742
weighted avg      0.945     0.947     0.945       742


=== Horizon t+2 ===
Confusion Matrix:
[[632  16]
 [ 40  54]]

Classification Report:
              precision    recall  f1-score   support

           0      0.940     0.975     0.958       648
           1      0.771     0.574     0.659        94

    accuracy                          0.925       742
   macro avg      0.856     0.775     0.808       742
weighted avg      0.919     0.925     0.920       742


=== Horizon t+3 ===
Confusion Matrix:
[[631  17]
 [ 47  47]]

Classification Report:
              preci